# Processing the Data for Labeling

In [ ]:
import re
import pandas as pd
from nltk.tokenize import sent_tokenize
import nltk
import os

In [18]:
books = {

    "SofS": {
        "file": "text_files/SofS.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2},\s*\d{4}
            \b
        """
    },

    "FitS": {
        "file": "text_files/FitS.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2}
            \b
        """
    },

    "DRtoF": {
        "file": "text_files/drtof.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b(
                # Full date with weekday (handles broken lines)
                (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
                (January|February|March|April|May|June|July|August|September|
                October|November|December)
                \s+\d{1,2}
                (?:\s*\n?\s*(?:st|nd|rd|th))?
                [\s,\n]*\d{4}

                |

                # Month + Year (January 1864)
                (January|February|March|April|May|June|July|August|September|
                October|November|December)
                \s+\d{4}

                |

                # Year only
                \b\d{4}\b
            )
        """
    },

    "PasWide": {
        "file": "text_files/PasWide.txt",
        "start_marker": "===== 9781443113342_epub_c01.htm =====",
        "end_marker": "===== 9781443113342_epub_cop_r1.htm =====",
        "chapter_pattern": r"===== 9781443113342_epub_c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2}
            \b
        """
    },

    "NbutC": {
        "file": "text_files/nbutc.txt",
        "start_marker": "===== ch1.html =====",
        "end_marker": "===== cop.html =====",
        "chapter_pattern": r"===== ch(\d+)\.html =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2}(?:st|nd|rd|th),\s*\d{4}
            \b
        """
    },

    "AinUTL": {
        "file": "text_files/AinUTL.txt",
        "start_marker": "===== ch1.html =====",
        "end_marker": "===== cop.html =====",
        "chapter_pattern": r"===== ch(\d+)\.html =====",
        "date_pattern": r"""
            \b
            le\s+\d{1,2}\s+
            (janvier|février|fevrier|mars|avril|mai|juin|juillet|août|aout|
             septembre|octobre|novembre|décembre|decembre)
            \s+\d{4}
            \b
        """
    }
}

In [21]:
# Process each book in the dictionary
book_dfs = {}

for book_name, book_info in books.items():
    with open(book_info["file"], "r", encoding="utf-8") as f:
        raw_text = f.read()

    start = raw_text.find(book_info["start_marker"])
    end = raw_text.find(book_info["end_marker"])
    diary_text = raw_text[start:end]

    sections = re.split(book_info["chapter_pattern"], diary_text)

    chapters = []
    for i in range(1, len(sections), 2):
        chapter_id = sections[i]
        text = sections[i+1]

        chapters.append({
            "book": book_name,
            "chapter_id": chapter_id,
            "text": text.strip()
        })

    chapters_df = pd.DataFrame(chapters)
    book_dfs[book_name] = chapters_df
    print(f"Chapters for {book_name}:")
    print(chapters_df.head())

Chapters for SofS:
   book chapter_id                                               text
0  SofS        c01  Dear Canada: A Sea of Sorrows\n\n\n\n\n\n\n\n\...
1  SofS        c02  Dear Canada: A Sea of Sorrows\n\n\n\n\n\n\n\n\...
2  SofS        c03  Dear Canada: A Sea of Sorrows\n\n\n\n\n\n\n\n\...
3  SofS        c04  Dear Canada: A Sea of Sorrows\n\n\n\n\n\n\n\n\...
4  SofS        c05  Dear Canada: A Sea of Sorrows\n\n\n\n\n\n\n\n\...
Chapters for FitS:
   book chapter_id                                               text
0  FitS        c01  Dear Canada: Footsteps in the Snow\n\n\n\n\n\n...
1  FitS        c02  Dear Canada: Footsteps in the Snow\n\n\n\n\n\n...
2  FitS        c03  Dear Canada: Footsteps in the Snow\n\n\n\n\n\n...
3  FitS        c04  Dear Canada: Footsteps in the Snow\n\n\n\n\n\n...
4  FitS        c05  Dear Canada: Footsteps in the Snow\n\n\n\n\n\n...
Chapters for DRtoF:
    book chapter_id                                               text
0  DRtoF        c01  Dear Canad

This data is pretty clean, because it comes from a a quality digital first source. Front and back matter were already removed in the previous section, but there are many many dates (at the start of each diary entry) and chapter formatting which need to be removed. 

In [15]:
def is_valid_sentence(s):
    if len(s) < 7:
        return False

    bad_endings = ("Mrs.", "Mr.", "Miss.", "Dr.", "“", '"')
    if s.endswith(bad_endings):
        return False

    return True

In [ ]:
# Creating entry and sentence DataFrames

base_output_folder = "dataframes"
os.makedirs(base_output_folder, exist_ok=True)

entry_output_folder = os.path.join(base_output_folder, "entry_dfs")
sentence_output_folder = os.path.join(base_output_folder, "sentence_dfs")

os.makedirs(entry_output_folder, exist_ok=True)
os.makedirs(sentence_output_folder, exist_ok=True)



for book_name, book_info in books.items():
    with open(book_info["file"], "r", encoding="utf-8") as f:
        raw_text = f.read()

    start = raw_text.find(book_info["start_marker"])
    end = raw_text.find(book_info["end_marker"])
    diary_text = raw_text[start:end]

    sections = re.split(book_info["chapter_pattern"], diary_text)

    all_entries = []
    all_sentences = []

    for i in range(1, len(sections), 2):
        chapter_id = sections[i]
        text = sections[i+1]

        # ---------------------------------------------------------
        # 1. Remove chapter header
        # ---------------------------------------------------------
        text = re.sub(
            r"^Dear Canada:.*?\n+",
            "",
            text,
            flags=re.DOTALL
        ).strip()

        # ---------------------------------------------------------
        # 2. Extract dated entries
        # ---------------------------------------------------------
        matches = list(re.finditer(book_info["date_pattern"], text, re.VERBOSE | re.IGNORECASE))

        for j, match in enumerate(matches):

            date = match.group()
            start = match.end()
            end = matches[j + 1].start() if j + 1 < len(matches) else len(text)

            entry_text = text[start:end]

            # -----------------------------------------------------
            # 3. Clean entry text
            # -----------------------------------------------------
            entry_text = re.sub(r"\n\s*\n", " ", entry_text)
            entry_text = re.sub(r"\s+", " ", entry_text).strip()

            entry_id = f"ch{chapter_id}_e{j+1}"

            all_entries.append({
                "book": book_name,
                "chapter_id": chapter_id,
                "entry_id": entry_id,
                "date": date,
                "entry_text": entry_text
            })

            # -----------------------------------------------------
            # 4. Sentence tokenize
            # -----------------------------------------------------
            sentences = sent_tokenize(entry_text)

            for sentence in sentences:

                sentence = sentence.strip()

                if is_valid_sentence(sentence):
                    all_sentences.append({
                        "book_name": book_name,
                        "chapter_id": chapter_id,
                        "entry_id": entry_id,
                        "date": date,
                        "sentence": sentence
                    })

    entries_df = pd.DataFrame(all_entries)
    sentences_df = pd.DataFrame(all_sentences)

    entries_df.to_csv(os.path.join(entry_output_folder, f"{book_name}_entries.csv"), index=False)
    sentences_df.to_csv(os.path.join(sentence_output_folder, f"{book_name}_sentences.csv"), index=False)

print(f"Diary entry and sentence DataFrames saved to folder: {entry_output_folder} and {sentence_output_folder}")

Diary entry and sentence DataFrames saved to folder: sentence_dfs


At this point, I went I tagged many, many sentences. The google sheets are linked here:

* A Sea of Sorrows: 
    * https://docs.google.com/spreadsheets/d/1wD1FXjlnSc4dNbAMG-ao3kaJ703kUrh9fN_qmiUsESQ/edit?usp=sharing

* With Nothing but Our Courage:
    * https://docs.google.com/spreadsheets/d/1mNxHFrLdaTxogkuXRKXPAowVM5RVcLqj4_0_fg9J7JY/edit?usp=sharing

* Alone in an Untamed Land: 
    * https://docs.google.com/spreadsheets/d/11Oo0RVQLp67KDq-6aKBtkG4sJUH1M-a48dOulzs0nRw/edit?usp=sharing 

* A Desperate Road to Freedom: 
    * https://docs.google.com/spreadsheets/d/1kUvav-Nw_3gy3ovdhktnaEf96SWWYTk-TzwVFTJ07IY/edit?usp=sharing

* Footsteps in the Snow
    * https://docs.google.com/spreadsheets/d/1lgwuJVaU-7Kv44IyCcicrF-7f_u8I6FdE7L4DFfPrq4/edit?usp=sharing

* A Prarie as Wide as the Sea
    * https://docs.google.com/spreadsheets/d/1fY5A0q57KMEGZ7heqP8C_8vT5zov41t6owsUj_uwfKg/edit?usp=sharing 